In [1]:
print("HELLO")

HELLO


In [4]:
import os
from pathlib import Path

cwd = Path(os.getcwd())
DOCS = Path(cwd.parent, "docs")

DOCS

WindowsPath('c:/Users/tejas/Desktop/Code/Hackathon/Buildonomics/AI/docs')

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader


file1 = Path(DOCS, "GS.pdf")
loader = PyMuPDFLoader(file_path=file1)

c:\Users\tejas\Desktop\Code\Hackathon\Buildonomics\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
RCT = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

text = loader.load()
texts = RCT.split_text(text[0].page_content)

In [35]:
type(text[0])

langchain_core.documents.base.Document

In [ ]:
from langchain_core.documents import Document

resume = [Document(page_content=text) for text in texts]
resume = text[0]


[Document(metadata={}, page_content='TEJAS KADAM\nSoftware Engineer Intern | Quantitative & AI Systems\nEmail: tejaskadam209@gmail.com\n|\nPhone: +91 8591877007\n|\nGitHub: github.com/Tejasisnothere\n|\nLinkedIn: linkedin.com/in/tejas-kadam2004'),
 Document(metadata={}, page_content='|\nLeetCode: leetcode.com/u/Tejasisnothere\nProfessional Summary\nB.Tech Computer Science and Business Systems student at Vellore Institute of Technology with a strong foundation'),
 Document(metadata={}, page_content='in Data Structures, Algorithms, and quantitative problem-solving (400+ LeetCode problems solved, JEE Percentile:'),
 Document(metadata={}, page_content='96.8). Experienced in building analytical and AI-driven systems for financial and forecasting use cases, including a'),
 Document(metadata={}, page_content='compliance-analysis engine and a time-series demand-forecasting platform, alongside multi-agent AI pipelines using'),
 Document(metadata={}, page_content='LangChain and LangGraph. Comfor

In [102]:
text[0]

Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-30T20:48:32+00:00', 'source': 'c:\\Users\\tejas\\Desktop\\Code\\Hackathon\\Buildonomics\\AI\\docs\\GS.pdf', 'file_path': 'c:\\Users\\tejas\\Desktop\\Code\\Hackathon\\Buildonomics\\AI\\docs\\GS.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-07-30T20:48:32+00:00', 'trapped': '', 'modDate': 'D:20260730204832Z', 'creationDate': 'D:20260730204832Z', 'page': 0}, page_content='TEJAS KADAM\nSoftware Engineer Intern | Quantitative & AI Systems\nEmail: tejaskadam209@gmail.com\n|\nPhone: +91 8591877007\n|\nGitHub: github.com/Tejasisnothere\n|\nLinkedIn: linkedin.com/in/tejas-kadam2004\n|\nLeetCode: leetcode.com/u/Tejasisnothere\nProfessional Summary\nB.Tech Computer Science and Business Systems student at Vellore Institute of Technology with a strong foundation\nin Data Structures, Algorithms, and quantitative problem-

In [41]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"))

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes = [
    "Person", "Institution", "Degree", "Skill",
    "Project", "Certification", "Organization", "Achievement",
    ],
allowed_relationships= [
    ("Person", "STUDIED_AT", "Institution"),
    ("Person", "HOLDS_DEGREE", "Degree"),
    ("Person", "HAS_SKILL", "Skill"),
    ("Person", "BUILT", "Project"),
    ("Project", "USES_TECHNOLOGY", "Skill"),
    ("Person", "EARNED_CERTIFICATION", "Certification"),
    ("Certification", "ISSUED_BY", "Organization"),
    ("Person", "ACHIEVED", "Achievement"),
]
 ,
)
graph_documents = llm_transformer.convert_to_graph_documents(resume)

In [42]:
from langchain_neo4j import Neo4jGraph


graph_store = Neo4jGraph(url="neo4j://127.0.0.1:7687", username="neo4j", password="Tjtk2004!", database="test")
graph_store.add_graph_documents(graph_documents=graph_documents)

In [43]:
graph_documents

[GraphDocument(nodes=[Node(id='Tejas Kadam', type='Person', properties={})], relationships=[], source=Document(metadata={}, page_content='TEJAS KADAM\nSoftware Engineer Intern | Quantitative & AI Systems\nEmail: tejaskadam209@gmail.com\n|\nPhone: +91 8591877007\n|\nGitHub: github.com/Tejasisnothere\n|\nLinkedIn: linkedin.com/in/tejas-kadam2004')),
 GraphDocument(nodes=[Node(id='Tejasisnothere', type='Person', properties={}), Node(id='Vellore Institute Of Technology', type='Institution', properties={}), Node(id='B.Tech Computer Science And Business Systems', type='Degree', properties={})], relationships=[Relationship(source=Node(id='Tejasisnothere', type='Person', properties={}), target=Node(id='Vellore Institute Of Technology', type='Institution', properties={}), type='STUDIED_AT', properties={}), Relationship(source=Node(id='Tejasisnothere', type='Person', properties={}), target=Node(id='B.Tech Computer Science And Business Systems', type='Degree', properties={}), type='HOLDS_DEGREE',

In [50]:
"""
Hardened resume -> knowledge graph pipeline.

Fixes applied vs. the earlier version:
  1. Explicit (source, relation, target) tuples in allowed_relationships,
     so the LLM can't emit orphan edges that skip Person.
  2. A canonical "anchor_id" injected into the prompt so every extraction
     pass ties back to the SAME Person node (prevents duplicate Person nodes
     across resume vs. README extraction runs).
  3. Name normalization before writing, so "Tejas Kadam" / "TEJAS KADAM" /
     " tejas kadam " never create separate nodes.
  4. add_graph_documents uses MERGE semantics under the hood via Neo4jGraph,
     but only if node ids match exactly post-normalization -- so normalization
     has to happen BEFORE convert_to_graph_documents, not after.
  5. A post-write connectivity check that flags any node not reachable from
     Person, so silent orphans get caught immediately instead of discovered
     later by eyeballing the graph.
"""

import os
import re
import json
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph

load_dotenv()


# ---------------------------------------------------------------------------
# 1. Normalization -- do this BEFORE any extraction, not after
# ---------------------------------------------------------------------------
def normalize_text(text: str) -> str:
    """Collapse whitespace only -- do NOT lowercase the whole resume,
    since that would mangle proper nouns like project/company names
    that the LLM uses verbatim as node ids."""
    return re.sub(r"\s+", " ", text).strip()


def canonical_person_name(raw_name: str) -> str:
    """Normalize just the person's name to a single consistent form,
    used to force every extraction pass to name the Person node identically."""
    return " ".join(word.capitalize() for word in raw_name.strip().split())


# ---------------------------------------------------------------------------
# 2. Schema: explicit (source, relation, target) triples
# ---------------------------------------------------------------------------
ALLOWED_NODES = [
    "Person", "Institution", "Degree", "Skill",
    "Project", "Certification", "Organization", "Achievement",
]

ALLOWED_RELATIONSHIPS = [
    ("Person", "STUDIED_AT", "Institution"),
    ("Person", "HOLDS_DEGREE", "Degree"),
    ("Person", "HAS_SKILL", "Skill"),
    ("Person", "BUILT", "Project"),
    ("Project", "USES_TECHNOLOGY", "Skill"),
    ("Person", "EARNED_CERTIFICATION", "Certification"),
    ("Certification", "ISSUED_BY", "Organization"),
    ("Person", "ACHIEVED", "Achievement"),
]


def build_transformer(llm, anchor_name: str) -> LLMGraphTransformer:
    """anchor_name gets baked into the additional_instructions so every
    extraction pass -- resume AND later README passes -- refers to the
    same Person node id, instead of letting the LLM re-derive the name
    from context each time (which is what caused the duplicate Person)."""
    return LLMGraphTransformer(
        llm=llm,
        allowed_nodes=ALLOWED_NODES,
        allowed_relationships=ALLOWED_RELATIONSHIPS,
        # node_properties=["description"],  # lets Project nodes carry a description property
        additional_instructions=(
            f"The central Person in this document is always named exactly "
            f"'{anchor_name}'. Use this exact string as the Person node's id "
            f"in every relationship. Every other node must connect back to "
            f"this Person, directly or indirectly -- do not emit a relationship "
            f"between two non-Person nodes without also connecting at least one "
            f"of them to Person elsewhere in your output."
        ),
    )


# ---------------------------------------------------------------------------
# 3. Extraction
# ---------------------------------------------------------------------------
def extract_resume_graph(resume_text: str, person_name: str, llm) -> list:
    anchor = canonical_person_name(person_name)
    text = normalize_text(resume_text)
    doc = Document(page_content=text, metadata={"source": "resume", "person": anchor})

    transformer = build_transformer(llm, anchor)
    return transformer.convert_to_graph_documents([doc])


# ---------------------------------------------------------------------------
# 4. Write -- MERGE-safe because Neo4jGraph.add_graph_documents already
#    uses MERGE on node id under the hood, so as long as ids are normalized
#    consistently (step 1-3), re-running this is idempotent.
# ---------------------------------------------------------------------------
def write_graph(graph: Neo4jGraph, graph_documents: list):
    graph.add_graph_documents(
        graph_documents,
        include_source=True,
        baseEntityLabel=True,
    )


# ---------------------------------------------------------------------------
# 5. Post-write validation -- catches orphans and duplicate Person nodes
#    immediately instead of relying on eyeballing the Explore view.
# ---------------------------------------------------------------------------
def validate_graph(graph: Neo4jGraph, person_name: str):
    anchor = canonical_person_name(person_name)

    # a) duplicate Person check
    dupes = graph.query(
        "MATCH (p:Person) RETURN p.id AS id, count(*) AS c"
    )
    person_nodes = [r for r in dupes if r["id"]]
    if len(person_nodes) > 1:
        print(f"WARNING: found {len(person_nodes)} Person nodes, expected 1:")
        for r in person_nodes:
            print(f"  - {r['id']}")

    # b) orphan check -- anything not reachable from Person
    orphans = graph.query(
        """
        MATCH (p:Person {id: $name})
        CALL (p) {
          MATCH (p)-[*]-(reachable)
          RETURN collect(DISTINCT reachable) AS reached
        }
        MATCH (n)
        WHERE NOT n IN reached AND n <> p AND n:Person = false
        RETURN labels(n) AS labels, n.id AS id
        """,
        params={"name": anchor},
    )
    if orphans:
        print(f"WARNING: {len(orphans)} node(s) not connected to Person:")
        for r in orphans:
            print(f"  - {r['labels']}: {r['id']}")
    else:
        print("Graph connectivity OK -- all nodes reachable from Person.")


# ---------------------------------------------------------------------------
# 6. Full run
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    llm = ChatGroq(model="openai/gpt-oss-120b")

    graph = Neo4jGraph(
        url=os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687"),
        username=os.environ.get("NEO4J_USERNAME", "neo4j"),
        password="Tjtk2004!",
        database=os.environ.get("NEO4J_DATABASE", "test"),
        refresh_schema=False,
    )

    resume_text = text[0].page_content   # your PDF-extracted text[0].page_content goes here
    person_name = "Tejas Kadam"  # exact name as it should appear as the anchor

    graph_documents = extract_resume_graph(resume_text, person_name, llm)
    write_graph(graph, graph_documents)
    validate_graph(graph, person_name)

BadRequestError: Error code: 400 - {'error': {'message': 'Failed to parse tool call arguments as JSON', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "DynamicGraph", "arguments": {\n  "nodes": [\n    {"id": "Tejas Kadam", "type": "Person"},\n    {"id": "Vellore Institute of Technology", "type": "Institution"},\n    {"id": "Ramsheth Thakur Public School", "type": "Institution"},\n    {"id": "Bachelor of Technology in Computer Science and Business Systems", "type": "Degree"},\n    {"id": "Python", "type": "Skill"},\n    {"id": "C++", "type": "Skill"},\n    {"id": "SQL", "type": "Skill"},\n    {"id": "JavaScript", "type": "Skill"},\n    {"id": "Data Structures & Algorithms", "type": "Skill"},\n    {"id": "OOP", "type": "Skill"},\n    {"id": "System Design", "type": "Skill"},\n    {"id": "Full-Stack Development", "type": "Skill"},\n    {"id": "Time-Series Forecasting", "type": "Skill"},\n    {"id": "NLP", "type": "Skill"},\n    {"id": "Reinforce

In [51]:
"""
Pydantic models for each node type in the resume knowledge graph.

These mirror the ALLOWED_NODES schema used in the LLMGraphTransformer
pipeline (resume_graph_pipeline_hardened.py). Use these when you want
structured, validated extraction instead of -- or alongside -- the
LLMGraphTransformer's free-form graph output. Handy for:
  - Validating LLM JSON output before writing to Neo4j
  - Type-safe access when reading nodes back out of the graph
  - A stricter alternative extraction path (structured output via
    llm.with_structured_output(...)) if LLMGraphTransformer keeps
    producing inconsistent shapes.
"""

from __future__ import annotations
from typing import Optional
from pydantic import BaseModel, Field


class Person(BaseModel):
    id: str = Field(..., description="Full name, e.g. 'Tejas Kadam'")
    email: Optional[str] = None
    phone: Optional[str] = None
    github: Optional[str] = None
    linkedin: Optional[str] = None
    leetcode: Optional[str] = None
    summary: Optional[str] = Field(None, description="Professional summary blurb")


class Institution(BaseModel):
    id: str = Field(..., description="Institution name, e.g. 'Vellore Institute of Technology'")
    location: Optional[str] = None


class Degree(BaseModel):
    id: str = Field(..., description="Degree name, e.g. 'B.Tech Computer Science and Business Systems'")
    institution_id: str = Field(..., description="id of the Institution this degree was earned at")
    start_year: Optional[int] = None
    end_year: Optional[int] = None
    cgpa: Optional[float] = None
    percentage: Optional[float] = None


class Skill(BaseModel):
    id: str = Field(..., description="Skill/tool/technology name, e.g. 'LangChain'")
    category: Optional[str] = Field(
        None, description="e.g. 'Language', 'Framework', 'Concept', 'Tool'"
    )


class Project(BaseModel):
    id: str = Field(..., description="Project name, e.g. 'NeuralNote'")
    description: Optional[str] = Field(None, description="1-2 sentence summary")
    tech_stack: list[str] = Field(default_factory=list, description="Skill ids used in this project")
    github_url: Optional[str] = None


class Certification(BaseModel):
    id: str = Field(..., description="Certification name")
    organization_id: str = Field(..., description="id of the Organization that issued it")
    certificate_code: Optional[str] = None
    hours: Optional[int] = None


class Organization(BaseModel):
    id: str = Field(..., description="Organization name, e.g. 'IBM'")


class Achievement(BaseModel):
    id: str = Field(..., description="Achievement description, e.g. 'Hackathon Finalist (3x)'")
    metric: Optional[str] = Field(None, description="Numeric or percentile value if applicable")


class ResumeGraph(BaseModel):
    """Top-level container -- the full structured extraction for one resume."""
    person: Person
    institutions: list[Institution] = Field(default_factory=list)
    degrees: list[Degree] = Field(default_factory=list)
    skills: list[Skill] = Field(default_factory=list)
    projects: list[Project] = Field(default_factory=list)
    certifications: list[Certification] = Field(default_factory=list)
    organizations: list[Organization] = Field(default_factory=list)
    achievements: list[Achievement] = Field(default_factory=list)

In [ ]:
"""
Resume -> Neo4j pipeline using structured output (Pydantic schema) instead of
LLMGraphTransformer's free-form JSON extraction.

Why this instead of LLMGraphTransformer:
  - The model's output is validated against the ResumeGraph schema before
    you ever touch it -- malformed/truncated output raises a clear
    validation error instead of the opaque "Failed to parse tool call
    arguments as JSON" BadRequestError you hit earlier.
  - Relationships (Degree -> Institution, Certification -> Organization)
    are enforced by the schema itself (institution_id / organization_id
    fields), not left to the LLM to remember to emit as separate edges.
  - Writing to Neo4j is done with explicit, predictable MERGE statements
    per node type, so it's easy to see exactly what graph shape you get --
    no surprises from the LLM inventing extra node/relationship types.

Install:
  pip install langchain-groq neo4j python-dotenv pydantic
"""

import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_neo4j import Neo4jGraph



load_dotenv()


# ---------------------------------------------------------------------------
# 1. Structured extraction
# ---------------------------------------------------------------------------
def extract_resume(resume_text: str, llm) -> ResumeGraph:
    structured_llm = llm.with_structured_output(ResumeGraph)
    prompt = (
        "Extract all information from this resume into the given schema. "
        "Use the person's exact name (as it appears at the top of the resume) "
        "as the Person id. Every Degree must reference a valid institution_id "
        "that also appears in the institutions list. Every Certification must "
        "reference a valid organization_id that also appears in the "
        "organizations list. Every Project's tech_stack must only contain "
        "skill ids that also appear in the skills list. Do not invent "
        "information that isn't in the resume text.\n\n"
        f"Resume:\n{resume_text}"
    )
    result = structured_llm.invoke(prompt)
    if not isinstance(result, ResumeGraph):
        # some provider/model combos return a dict instead of the model instance
        result = ResumeGraph.model_validate(result)
    return result


# ---------------------------------------------------------------------------
# 2. Write to Neo4j -- explicit MERGE per node type, then relationships.
#    MERGE (not CREATE) makes every write idempotent: re-running this on
#    the same resume updates existing nodes instead of duplicating them.
# ---------------------------------------------------------------------------
def write_resume_graph(graph: Neo4jGraph, data: ResumeGraph):
    
    graph.query(
        """
        MERGE (p:Person {id: $id})
        SET p.email = $email, p.phone = $phone, p.github = $github,
            p.linkedin = $linkedin, p.leetcode = $leetcode, p.summary = $summary
        """,
        params=data.person.model_dump(),
    )

    for inst in data.institutions:
        graph.query(
            """
            MERGE (i:Institution {id: $id})
            SET i.location = $location
            WITH i
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:STUDIED_AT]->(i)
            """,
            params={**inst.model_dump(), "person_id": data.person.id},
        )

    for deg in data.degrees:
        graph.query(
            """
            MERGE (d:Degree {id: $id})
            SET d.start_year = $start_year, d.end_year = $end_year,
                d.cgpa = $cgpa, d.percentage = $percentage
            WITH d
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:HOLDS_DEGREE]->(d)
            WITH d
            MATCH (i:Institution {id: $institution_id})
            MERGE (d)-[:AT_INSTITUTION]->(i)
            """,
            params={**deg.model_dump(), "person_id": data.person.id},
        )

    for skill in data.skills:
        graph.query(
            """
            MERGE (s:Skill {id: $id})
            SET s.category = $category
            WITH s
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:HAS_SKILL]->(s)
            """,
            params={**skill.model_dump(), "person_id": data.person.id},
        )

    
    for proj in data.projects:
        graph.query(
            """
            MERGE (pr:Project {id: $id})
            SET pr.description = $description, pr.github_url = $github_url
            WITH pr
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:BUILT]->(pr)
            WITH pr
            UNWIND $tech_stack AS tech_id
            MATCH (s:Skill {id: tech_id})
            MERGE (pr)-[:USES_TECHNOLOGY]->(s)
            """,
            params={**proj.model_dump(), "person_id": data.person.id},
        )

    
    for org in data.organizations:
        graph.query("MERGE (o:Organization {id: $id})", params=org.model_dump())

    for cert in data.certifications:
        graph.query(
            """
            MERGE (c:Certification {id: $id})
            SET c.certificate_code = $certificate_code, c.hours = $hours
            WITH c
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:EARNED_CERTIFICATION]->(c)
            WITH c
            MATCH (o:Organization {id: $organization_id})
            MERGE (c)-[:ISSUED_BY]->(o)
            """,
            params={**cert.model_dump(), "person_id": data.person.id},
        )

    for ach in data.achievements:
        graph.query(
            """
            MERGE (a:Achievement {id: $id})
            SET a.metric = $metric
            WITH a
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:ACHIEVED]->(a)
            """,
            params={**ach.model_dump(), "person_id": data.person.id},
        )


def validate_graph(graph: Neo4jGraph, person_id: str):
    dupes = graph.query("MATCH (p:Person) RETURN p.id AS id")
    if len(dupes) > 1:
        print(f"WARNING: found {len(dupes)} Person nodes, expected 1:")
        for r in dupes:
            print(f"  - {r['id']}")

    orphans = graph.query(
        """
        MATCH (p:Person {id: $id})
        CALL (p) {
          MATCH (p)-[*]-(reachable)
          RETURN collect(DISTINCT reachable) AS reached
        }
        MATCH (n)
        WHERE NOT n IN reached AND n <> p AND NOT n:Person
        RETURN labels(n) AS labels, n.id AS id
        """,
        params={"id": person_id},
    )
    if orphans:
        print(f"WARNING: {len(orphans)} node(s) not connected to Person:")
        for r in orphans:
            print(f"  - {r['labels']}: {r['id']}")
    else:
        print("Graph connectivity OK -- all nodes reachable from Person.")


if __name__ == "__main__":
    llm = ChatGroq(model="openai/gpt-oss-120b", max_tokens=8000)

    graph = Neo4jGraph(
        url=os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687"),
        username=os.environ.get("NEO4J_USERNAME", "neo4j"),
        password="Tjtk2004!",
        database=os.environ.get("NEO4J_DATABASE", "test"),
        refresh_schema=False,
    )

    resume_text =  text[0].page_content # your PDF-extracted text[0].page_content goes here

    data = extract_resume(resume_text, llm)
    print(data.model_dump_json(indent=2))  # inspect before writing, if you want

    write_resume_graph(graph, data)
    validate_graph(graph, data.person.id)

{
  "person": {
    "id": "TEJAS KADAM",
    "email": "tejaskadam209@gmail.com",
    "phone": "+91 8591877007",
    "github": "github.com/Tejasisnothere",
    "linkedin": "linkedin.com/in/tejas-kadam2004",
    "leetcode": "leetcode.com/u/Tejasisnothere",
    "summary": "B.Tech Computer Science and Business Systems student at Vellore Institute of Technology with a strong foundation in Data Structures, Algorithms, and quantitative problem-solving (400+ LeetCode problems solved, JEE Percentile: 96.8). Experienced in building analytical and AI-driven systems for financial and forecasting use cases, including a compliance-analysis engine and a time-series demand-forecasting platform, alongside multi-agent AI pipelines using LangChain and LangGraph. Comfortable combining rigorous engineering with data-driven decision support."
  },
  "institutions": [
    {
      "id": "Vellore Institute of Technology",
      "location": null
    }
  ],
  "degrees": [
    {
      "id": "Bachelor of Technolog

In [56]:
"""
Resume extraction split into one function per node type, instead of one
giant ResumeGraph structured-output call.

Why split it up:
  - Smaller per-call output = far less risk of the truncation issue you
    hit earlier ("Failed to parse tool call arguments as JSON") -- each
    call now only has to produce one list, not the whole graph at once.
  - Easier to debug: if Projects come out wrong, you can re-run just
    extract_projects() without re-extracting everything else.
  - Cross-referencing (Degree -> Institution, Project -> Skill) is done
    by passing already-extracted lists into later calls as context, so
    the LLM has a fixed vocabulary to pick ids from instead of inventing
    new ones each call.

Install: pip install langchain-groq pydantic
"""

import os
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_neo4j import Neo4jGraph


load_dotenv()


# ---------------------------------------------------------------------------
# Small list-wrapper models -- with_structured_output needs a single
# top-level model, so each extractor gets a thin wrapper around a list.
# ---------------------------------------------------------------------------
class InstitutionList(BaseModel):
    institutions: List[Institution] = Field(default_factory=list)

class DegreeList(BaseModel):
    degrees: List[Degree] = Field(default_factory=list)

class SkillList(BaseModel):
    skills: List[Skill] = Field(default_factory=list)

class ProjectList(BaseModel):
    projects: List[Project] = Field(default_factory=list)

class CertificationList(BaseModel):
    certifications: List[Certification] = Field(default_factory=list)

class OrganizationList(BaseModel):
    organizations: List[Organization] = Field(default_factory=list)

class AchievementList(BaseModel):
    achievements: List[Achievement] = Field(default_factory=list)


# ---------------------------------------------------------------------------
# One extraction function per node type
# ---------------------------------------------------------------------------
def extract_person(resume_text: str, llm) -> Person:
    structured_llm = llm.with_structured_output(Person)
    prompt = (
        "Extract the candidate's identity info from this resume: full name "
        "(as the id), email, phone, github, linkedin, leetcode, and a 1-2 "
        "sentence professional summary. Leave fields as null if not present.\n\n"
        f"Resume:\n{resume_text}"
    )
    result = structured_llm.invoke(prompt)
    return result if isinstance(result, Person) else Person.model_validate(result)


def extract_institutions(resume_text: str, llm) -> List[Institution]:
    structured_llm = llm.with_structured_output(InstitutionList)
    prompt = (
        "Extract every school, college, or university mentioned in the "
        "Education section of this resume as Institution entries. Use the "
        "institution's short proper name as the id (not the degree name).\n\n"
        f"Resume:\n{resume_text}"
    )
    result = structured_llm.invoke(prompt)
    result = result if isinstance(result, InstitutionList) else InstitutionList.model_validate(result)
    return result.institutions


def extract_degrees(resume_text: str, institutions: List[Institution], llm) -> List[Degree]:
    inst_ids = [i.id for i in institutions]
    structured_llm = llm.with_structured_output(DegreeList)
    prompt = (
        "Extract every degree/qualification from the Education section as "
        "Degree entries (including school-level qualifications like Class X/XII "
        "if present). Each Degree.institution_id MUST be exactly one of these "
        f"existing institution ids -- do not invent a new one: {inst_ids}\n\n"
        f"Resume:\n{resume_text}"
    )
    result = structured_llm.invoke(prompt)
    result = result if isinstance(result, DegreeList) else DegreeList.model_validate(result)
    return result.degrees


def extract_skills(resume_text: str, llm) -> List[Skill]:
    structured_llm = llm.with_structured_output(SkillList)
    prompt = (
        "Extract every individual skill, language, framework, tool, or "
        "concept listed in the Technical Skills section as separate Skill "
        "entries. Split comma-separated lists into individual items -- do "
        "not group multiple skills into one id.\n\n"
        f"Resume:\n{resume_text}"
    )
    result = structured_llm.invoke(prompt)
    result = result if isinstance(result, SkillList) else SkillList.model_validate(result)
    return result.skills


def extract_projects(resume_text: str, skills: List[Skill], llm) -> List[Project]:
    skill_ids = [s.id for s in skills]
    structured_llm = llm.with_structured_output(ProjectList)
    prompt = (
        "Extract every project from the Projects section. Each project is "
        "listed as a short heading line (format like 'NAME — Subtitle'), "
        "followed by bullet points. Project.id MUST be ONLY the short NAME "
        "from the heading -- never a sentence or bullet-point fragment. Put "
        "a 1-2 sentence summary of the bullets in description instead. "
        "Project.tech_stack must only contain ids from this existing skill "
        f"list, matching technologies mentioned for that project: {skill_ids}\n\n"
        f"Resume:\n{resume_text}"
    )
    result = structured_llm.invoke(prompt)
    result = result if isinstance(result, ProjectList) else ProjectList.model_validate(result)
    return result.projects


def extract_organizations(resume_text: str, llm) -> List[Organization]:
    structured_llm = llm.with_structured_output(OrganizationList)
    prompt = (
        "Extract every organization/company that issued a certification "
        "mentioned in this resume (e.g. the issuing body, not the platform "
        "it was hosted on, unless both are named separately).\n\n"
        f"Resume:\n{resume_text}"
    )
    result = structured_llm.invoke(prompt)
    result = result if isinstance(result, OrganizationList) else OrganizationList.model_validate(result)
    return result.organizations


def extract_certifications(resume_text: str, organizations: List[Organization], llm) -> List[Certification]:
    org_ids = [o.id for o in organizations]
    structured_llm = llm.with_structured_output(CertificationList)
    prompt = (
        "Extract every certification from the Certifications section. Each "
        "Certification.organization_id MUST be exactly one of these existing "
        f"organization ids -- do not invent a new one: {org_ids}\n\n"
        f"Resume:\n{resume_text}"
    )
    result = structured_llm.invoke(prompt)
    result = result if isinstance(result, CertificationList) else CertificationList.model_validate(result)
    return result.certifications


def extract_achievements(resume_text: str, llm) -> List[Achievement]:
    structured_llm = llm.with_structured_output(AchievementList)
    prompt = (
        "Extract every achievement from the Achievements section as separate "
        "Achievement entries, one per bullet.\n\n"
        f"Resume:\n{resume_text}"
    )
    result = structured_llm.invoke(prompt)
    result = result if isinstance(result, AchievementList) else AchievementList.model_validate(result)
    return result.achievements


# ---------------------------------------------------------------------------
# Pipeline: run each extractor in order, feeding earlier results forward
# as the fixed vocabulary for later calls.
# ---------------------------------------------------------------------------
def run_extraction_pipeline(resume_text: str, llm) -> ResumeGraph:
    print("Extracting person...")
    person = extract_person(resume_text, llm)

    print("Extracting institutions...")
    institutions = extract_institutions(resume_text, llm)

    print("Extracting degrees...")
    degrees = extract_degrees(resume_text, institutions, llm)

    print("Extracting skills...")
    skills = extract_skills(resume_text, llm)

    print("Extracting projects...")
    projects = extract_projects(resume_text, skills, llm)

    print("Extracting organizations...")
    organizations = extract_organizations(resume_text, llm)

    print("Extracting certifications...")
    certifications = extract_certifications(resume_text, organizations, llm)

    print("Extracting achievements...")
    achievements = extract_achievements(resume_text, llm)

    return ResumeGraph(
        person=person,
        institutions=institutions,
        degrees=degrees,
        skills=skills,
        projects=projects,
        certifications=certifications,
        organizations=organizations,
        achievements=achievements,
    )


# ---------------------------------------------------------------------------
# Full run: extract -> write -> validate
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    llm = ChatGroq(model="openai/gpt-oss-120b", max_tokens=4000)

    graph = Neo4jGraph(
        url=os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687"),
        username=os.environ.get("NEO4J_USERNAME", "neo4j"),
        password="Tjtk2004!",
        database=os.environ.get("NEO4J_DATABASE", "test"),
        refresh_schema=False,
    )

    resume_text = text[0].page_content  # your PDF-extracted text[0].page_content goes here

    data = run_extraction_pipeline(resume_text, llm)
    print(data.model_dump_json(indent=2))

    write_resume_graph(graph, data)
    validate_graph(graph, data.person.id)

Extracting person...
Extracting institutions...
Extracting degrees...
Extracting skills...
Extracting projects...
Extracting organizations...


BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'IBM  \nAdroit ProLearn Technologies  \nUdemy  \nKrish Naik  \nKRISHAI Technologies'}}

In [61]:
from pydantic import Field
class Extractor(BaseModel):
    sections: dict[str, str] = Field(
        description="Map each resume section name to its complete original textual content."
    )

In [64]:
from typing import Dict
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


class Extractor(BaseModel):
    sections: Dict[str, str] = Field(
        description="Each key is a resume section name and each value is the complete original text belonging to that section."
    )


new_llm = llm.with_structured_output(
    Extractor,
    method="json_schema",
)

prompt = ChatPromptTemplate.from_template("""
Read the resume and divide it into its different sections.

For every section:
- Use the section's name as the key.
- Put the COMPLETE ORIGINAL TEXT belonging to that section as the value.
- Do NOT summarize.
- Do NOT paraphrase.
- Do NOT extract individual entities.
- Preserve every project, experience entry, certification, education entry,
  bullet point, technology, date, and other information belonging to that section.
- Determine where a section starts and ends from the resume structure.
- Do not move content between sections.

For example, if there are 4 projects, the "Projects" value must contain
ALL 4 projects and ALL of their associated text.

Resume:
{resume}
""")

chain = prompt | new_llm

result = chain.invoke({
    "resume": text[0].page_content
})

print(result)

sections={'Header': 'TEJAS KADAM\nSoftware Engineer Intern | Quantitative & AI Systems\nEmail: tejaskadam209@gmail.com\n|\nPhone: +91 8591877007\n|\nGitHub: github.com/Tejasisnothere\n|\nLinkedIn: linkedin.com/in/tejas-kadam2004\n|\nLeetCode: leetcode.com/u/Tejasisnothere', 'Professional Summary': 'B.Tech Computer Science and Business Systems student at Vellore Institute of Technology with a strong foundation\nin Data Structures, Algorithms, and quantitative problem-solving (400+ LeetCode problems solved, JEE Percentile:\n96.8). Experienced in building analytical and AI-driven systems for financial and forecasting use cases, including a\ncompliance-analysis engine and a time-series demand-forecasting platform, alongside multi-agent AI pipelines using\nLangChain and LangGraph. Comfortable combining rigorous engineering with data-driven decision support.', 'Education': 'Bachelor of Technology in Computer Science and Business Systems\n2024 – 2028\nVellore Institute of Technology\nCGPA: 8.

In [67]:
result.sections.keys()

dict_keys(['Header', 'Professional Summary', 'Education', 'Technical Skills', 'Experience', 'Projects', 'Certifications', 'Achievements'])

In [68]:
result.sections['Experience']

'Analytical & AI Systems Development (Self-driven)\nLast 3+ Months\n• Designed and built multi-agent, retrieval-augmented AI systems using LangChain and LangGraph, orchestrating\nplanning, routing, retrieval, and generation across specialized agents.\n• Built structured evaluation and retrieval-accuracy pipelines with Qdrant, applying dynamic query generation and\nmetadata extraction to improve precision of data-driven outputs.'

In [73]:
from typing import List, Optional
from pydantic import BaseModel, Field

class ExperienceEntry(BaseModel):
    company: str
    role: str
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    skills_used: List[str] = Field(default_factory=list)
    description: str

class ExperienceExtractor(BaseModel):
    entries: List[ExperienceEntry]

class EducationEntry(BaseModel):
    institution: str
    degree: str
    field_of_study: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None

class EducationExtractor(BaseModel):
    entries: List[EducationEntry]

class ProjectEntry(BaseModel):
    name: str
    description: str
    skills_used: List[str] = Field(default_factory=list)

class ProjectExtractor(BaseModel):
    entries: List[ProjectEntry]

class SkillsExtractor(BaseModel):
    skills: List[str]

class AchievementEntry(BaseModel):
    title: str
    description: Optional[str] = None
    date: Optional[str] = None
    issuer: Optional[str] = None  # e.g. "Certification issued by AWS", award-giving body, etc.

class AchievementExtractor(BaseModel):
    entries: List[AchievementEntry]

class OtherEntry(BaseModel):
    heading: str          # original section name, preserved as-is
    content: str           # raw/lightly cleaned text, unmodified

class OtherExtractor(BaseModel):
    entries: List[OtherEntry]

In [74]:
SECTION_SCHEMA_MAP = {
    "experience": ExperienceExtractor,
    "work experience": ExperienceExtractor,
    "professional experience": ExperienceExtractor,
    "education": EducationExtractor,
    "projects": ProjectExtractor,
    "skills": SkillsExtractor,
    "technical skills": SkillsExtractor,
    "achievements": AchievementExtractor,
    "awards": AchievementExtractor,
    "honors": AchievementExtractor,
    "certifications": AchievementExtractor,
    "certificates": AchievementExtractor,
    "licenses": AchievementExtractor,
}

def route_section(name: str):
    key = name.strip().lower()
    for k, schema in SECTION_SCHEMA_MAP.items():
        if k in key:
            return schema
    return OtherExtractor  # fallback instead of None — nothing gets dropped

extracted = {}
for section_name, section_text in result.sections.items():
    schema = route_section(section_name)
    if schema is OtherExtractor:
        section_llm = llm.with_structured_output(schema, method="json_schema")
        extracted.setdefault("Other", OtherExtractor(entries=[])).entries.append(
            OtherEntry(heading=section_name, content=section_text)
        )
        continue
    section_llm = llm.with_structured_output(schema, method="json_schema")
    extracted[section_name] = section_llm.invoke(
        f"Extract structured entries from this resume section:\n\n{section_text}"
    )

In [75]:
extracted.keys()

dict_keys(['Other', 'Education', 'Technical Skills', 'Experience', 'Projects', 'Certifications', 'Achievements'])

In [86]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

load_dotenv()

graph = Neo4jGraph(
    url=os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687"),
    username=os.environ.get("NEO4J_USERNAME", "neo4j"),
    password="Tjtk2004!",   # set in .env, never hardcode
    database=os.environ.get("NEO4J_DATABASE", "neo4j"),
    refresh_schema=False,
)


def write_person_graph(graph: Neo4jGraph, person_id: str, extracted: dict):
    graph.query("MERGE (p:Person {id: $person_id})", params={"person_id": person_id})

    # --- Experience ---
    for entry in extracted.get("Experience", ExperienceExtractor(entries=[])).entries:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (c:Company {name: $company})
            MERGE (r:Role {title: $role, company: $company})
            MERGE (p)-[rel:WORKED_AT]->(c)
            SET rel.role = $role, rel.start_date = $start, rel.end_date = $end
            MERGE (p)-[:HELD_ROLE]->(r)
            WITH p, r
            UNWIND $skills AS skill_name
            MERGE (s:Skill {name: toLower(skill_name)})
            MERGE (r)-[:USED_SKILL]->(s)
        """, params={
            "person_id": person_id, "company": entry.company, "role": entry.role,
            "start": entry.start_date, "end": entry.end_date, "skills": entry.skills_used
        })

    # --- Education ---
    for entry in extracted.get("Education", EducationExtractor(entries=[])).entries:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (i:Institution {name: $institution})
            MERGE (d:Degree {name: $degree})
            SET d.field = $field
            MERGE (p)-[rel:STUDIED_AT]->(i)
            SET rel.start_date = $start, rel.end_date = $end
            MERGE (p)-[:EARNED]->(d)
        """, params={
            "person_id": person_id, "institution": entry.institution, "degree": entry.degree,
            "field": entry.field_of_study, "start": entry.start_date, "end": entry.end_date
        })

    # --- Projects ---
    for entry in extracted.get("Projects", ProjectExtractor(entries=[])).entries:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (proj:Project {name: $name})
            SET proj.description = $description
            MERGE (p)-[:WORKED_ON]->(proj)
            WITH proj
            UNWIND $skills AS skill_name
            MERGE (s:Skill {name: toLower(skill_name)})
            MERGE (proj)-[:USED_SKILL]->(s)
        """, params={
            "person_id": person_id, "name": entry.name,
            "description": entry.description, "skills": entry.skills_used
        })

    # --- Skills ---
    for skill_name in extracted.get("Skills", SkillsExtractor(skills=[])).skills:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (s:Skill {name: toLower($skill_name)})
            MERGE (p)-[:HAS_SKILL]->(s)
        """, params={"person_id": person_id, "skill_name": skill_name})

    # --- Achievements ---
    for entry in extracted.get("Achievements", AchievementExtractor(entries=[])).entries:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (a:Achievement {title: $title})
            SET a.description = $description,
                a.date = $date,
                a.issuer = $issuer
            MERGE (p)-[:ACHIEVED]->(a)
        """, params={
            "person_id": person_id, "title": entry.title, "description": entry.description,
            "date": entry.date, "issuer": entry.issuer
        })

    # --- Other ---
    for entry in extracted.get("Other", OtherExtractor(entries=[])).entries:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (o:OtherSection {heading: $heading, person_id: $person_id})
            SET o.content = $content
            MERGE (p)-[:HAS_SECTION]->(o)
        """, params={
            "person_id": person_id, "heading": entry.heading, "content": entry.content
        })


write_person_graph(graph, "person_123", extracted)

In [85]:
extracted

{'Other': OtherExtractor(entries=[OtherEntry(heading='Header', content='TEJAS KADAM\nSoftware Engineer Intern | Quantitative & AI Systems\nEmail: tejaskadam209@gmail.com\n|\nPhone: +91 8591877007\n|\nGitHub: github.com/Tejasisnothere\n|\nLinkedIn: linkedin.com/in/tejas-kadam2004\n|\nLeetCode: leetcode.com/u/Tejasisnothere'), OtherEntry(heading='Professional Summary', content='B.Tech Computer Science and Business Systems student at Vellore Institute of Technology with a strong foundation\nin Data Structures, Algorithms, and quantitative problem-solving (400+ LeetCode problems solved, JEE Percentile:\n96.8). Experienced in building analytical and AI-driven systems for financial and forecasting use cases, including a\ncompliance-analysis engine and a time-series demand-forecasting platform, alongside multi-agent AI pipelines using\nLangChain and LangGraph. Comfortable combining rigorous engineering with data-driven decision support.')]),
 'Education': EducationExtractor(entries=[Education

In [89]:
import os
from typing import List, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_neo4j import Neo4jGraph

load_dotenv()

# ---------------------------------------------------------------------------
# 1. Schemas
# ---------------------------------------------------------------------------
class ExperienceEntry(BaseModel):
    company: str
    role: str
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    skills_used: List[str] = Field(default_factory=list)
    description: str

class ExperienceExtractor(BaseModel):
    entries: List[ExperienceEntry]

class EducationEntry(BaseModel):
    institution: str
    degree: str
    field_of_study: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None

class EducationExtractor(BaseModel):
    entries: List[EducationEntry]

class ProjectEntry(BaseModel):
    name: str
    description: str
    skills_used: List[str] = Field(default_factory=list)

class ProjectExtractor(BaseModel):
    entries: List[ProjectEntry]

class SkillsExtractor(BaseModel):
    skills: List[str]

class AchievementEntry(BaseModel):
    title: str
    description: Optional[str] = None
    date: Optional[str] = None
    issuer: Optional[str] = None

class AchievementExtractor(BaseModel):
    entries: List[AchievementEntry]

class OtherEntry(BaseModel):
    heading: str
    content: str

class OtherExtractor(BaseModel):
    entries: List[OtherEntry]


# ---------------------------------------------------------------------------
# 2. Routing: section name -> schema, AND schema -> canonical storage key
# ---------------------------------------------------------------------------
SECTION_SCHEMA_MAP = {
    "experience": ExperienceExtractor,
    "work experience": ExperienceExtractor,
    "professional experience": ExperienceExtractor,
    "education": EducationExtractor,
    "projects": ProjectExtractor,
    "skills": SkillsExtractor,
    "technical skills": SkillsExtractor,
    "achievements": AchievementExtractor,
    "awards": AchievementExtractor,
    "honors": AchievementExtractor,
    "certifications": AchievementExtractor,
    "certificates": AchievementExtractor,
    "licenses": AchievementExtractor,
}

CANONICAL_KEY = {
    ExperienceExtractor: "Experience",
    EducationExtractor: "Education",
    ProjectExtractor: "Projects",
    SkillsExtractor: "Skills",
    AchievementExtractor: "Achievements",
}

def route_section(name: str):
    key = name.strip().lower()
    for k, schema in SECTION_SCHEMA_MAP.items():
        if k in key:
            return schema
    return OtherExtractor


# ---------------------------------------------------------------------------
# 3. Extraction — writes to canonical keys, not raw headings
# ---------------------------------------------------------------------------
def extract_all_sections(result, llm) -> dict:
    extracted = {}
    for section_name, section_text in result.sections.items():
        schema = route_section(section_name)

        if schema is OtherExtractor:
            extracted.setdefault("Other", OtherExtractor(entries=[])).entries.append(
                OtherEntry(heading=section_name, content=section_text)
            )
            continue

        section_llm = llm.with_structured_output(schema, method="json_schema")
        parsed = section_llm.invoke(
            f"Extract structured entries from this resume section:\n\n{section_text}"
        )
        canonical_key = CANONICAL_KEY[schema]

        # merge if this canonical key was already populated by another
        # section heading (e.g. both "Certifications" and "Licenses" -> Achievements)
        if canonical_key in extracted:
            if hasattr(parsed, "entries"):
                extracted[canonical_key].entries.extend(parsed.entries)
            elif hasattr(parsed, "skills"):
                extracted[canonical_key].skills.extend(parsed.skills)
        else:
            extracted[canonical_key] = parsed

    return extracted


# ---------------------------------------------------------------------------
# 4. Graph connection
# ---------------------------------------------------------------------------
graph = Neo4jGraph(
    url=os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687"),
    username=os.environ.get("NEO4J_USERNAME", "neo4j"),
    password="Tjtk2004!",   # set in .env — rotate the old one
    database="test",
    refresh_schema=False,
)


# ---------------------------------------------------------------------------
# 5. Graph write
# ---------------------------------------------------------------------------
def write_person_graph(graph: Neo4jGraph, person_id: str, extracted: dict):
    graph.query("MERGE (p:Person {id: $person_id})", params={"person_id": person_id})

    for entry in extracted.get("Experience", ExperienceExtractor(entries=[])).entries:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (c:Company {name: $company})
            MERGE (r:Role {title: $role, company: $company})
            MERGE (p)-[rel:WORKED_AT]->(c)
            SET rel.role = $role, rel.start_date = $start, rel.end_date = $end
            MERGE (p)-[:HELD_ROLE]->(r)
            WITH p, r
            UNWIND $skills AS skill_name
            MERGE (s:Skill {name: toLower(skill_name)})
            MERGE (r)-[:USED_SKILL]->(s)
        """, params={
            "person_id": person_id, "company": entry.company, "role": entry.role,
            "start": entry.start_date, "end": entry.end_date, "skills": entry.skills_used
        })

    for entry in extracted.get("Education", EducationExtractor(entries=[])).entries:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (i:Institution {name: $institution})
            MERGE (d:Degree {name: $degree})
            SET d.field = $field
            MERGE (p)-[rel:STUDIED_AT]->(i)
            SET rel.start_date = $start, rel.end_date = $end
            MERGE (p)-[:EARNED]->(d)
        """, params={
            "person_id": person_id, "institution": entry.institution, "degree": entry.degree,
            "field": entry.field_of_study, "start": entry.start_date, "end": entry.end_date
        })

    for entry in extracted.get("Projects", ProjectExtractor(entries=[])).entries:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (proj:Project {name: $name})
            SET proj.description = $description
            MERGE (p)-[:WORKED_ON]->(proj)
            WITH proj
            UNWIND $skills AS skill_name
            MERGE (s:Skill {name: toLower(skill_name)})
            MERGE (proj)-[:USED_SKILL]->(s)
        """, params={
            "person_id": person_id, "name": entry.name,
            "description": entry.description, "skills": entry.skills_used
        })

    for skill_name in extracted.get("Skills", SkillsExtractor(skills=[])).skills:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (s:Skill {name: toLower($skill_name)})
            MERGE (p)-[:HAS_SKILL]->(s)
        """, params={"person_id": person_id, "skill_name": skill_name})

    for entry in extracted.get("Achievements", AchievementExtractor(entries=[])).entries:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (a:Achievement {title: $title})
            SET a.description = $description,
                a.date = $date,
                a.issuer = $issuer
            MERGE (p)-[:ACHIEVED]->(a)
        """, params={
            "person_id": person_id, "title": entry.title, "description": entry.description,
            "date": entry.date, "issuer": entry.issuer
        })

    for entry in extracted.get("Other", OtherExtractor(entries=[])).entries:
        graph.query("""
            MATCH (p:Person {id: $person_id})
            MERGE (o:OtherSection {heading: $heading, person_id: $person_id})
            SET o.content = $content
            MERGE (p)-[:HAS_SECTION]->(o)
        """, params={
            "person_id": person_id, "heading": entry.heading, "content": entry.content
        })


# ---------------------------------------------------------------------------
# 6. Run
# ---------------------------------------------------------------------------
extracted = extract_all_sections(result, llm)
print(extracted.keys())   # sanity check before writing

write_person_graph(graph, "person_123", extracted)

print(graph.query("MATCH (n) RETURN labels(n) AS labels, count(*) AS c"))

dict_keys(['Other', 'Education', 'Skills', 'Experience', 'Projects', 'Achievements'])
[{'labels': ['Person'], 'c': 1}, {'labels': ['Company'], 'c': 1}, {'labels': ['Role'], 'c': 1}, {'labels': ['Skill'], 'c': 35}, {'labels': ['Institution'], 'c': 2}, {'labels': ['Degree'], 'c': 3}, {'labels': ['Project'], 'c': 4}, {'labels': ['Achievement'], 'c': 5}, {'labels': ['OtherSection'], 'c': 2}]


In [90]:
graph.refresh_schema()

print(graph.schema)

Node properties:
Person {id: STRING}
Company {name: STRING}
Role {company: STRING, title: STRING}
Institution {name: STRING}
Degree {name: STRING, field: STRING}
Skill {name: STRING}
Achievement {description: STRING, title: STRING, issuer: STRING}
Project {name: STRING, description: STRING}
OtherSection {heading: STRING, person_id: STRING, content: STRING}
Relationship properties:
STUDIED_AT {start_date: STRING, end_date: STRING}
WORKED_AT {role: STRING}
The relationships:
(:Person)-[:STUDIED_AT]->(:Institution)
(:Person)-[:EARNED]->(:Degree)
(:Person)-[:WORKED_ON]->(:Project)
(:Person)-[:HAS_SECTION]->(:OtherSection)
(:Person)-[:ACHIEVED]->(:Achievement)
(:Person)-[:HAS_SKILL]->(:Skill)
(:Person)-[:WORKED_AT]->(:Company)
(:Person)-[:HELD_ROLE]->(:Role)
(:Role)-[:USED_SKILL]->(:Skill)
(:Project)-[:USED_SKILL]->(:Skill)


In [92]:
schema = """
Node properties:
Person {id: STRING}
Company {name: STRING}
Role {company: STRING, title: STRING}
Institution {name: STRING}
Degree {name: STRING, field: STRING}
Skill {name: STRING}
Achievement {description: STRING, title: STRING, issuer: STRING}
Project {name: STRING, description: STRING}
OtherSection {heading: STRING, person_id: STRING, content: STRING}

Relationship properties:
STUDIED_AT {start_date: STRING, end_date: STRING}
WORKED_AT {role: STRING}

Relationships:
(:Person)-[:STUDIED_AT]->(:Institution)
(:Person)-[:EARNED]->(:Degree)
(:Person)-[:WORKED_ON]->(:Project)
(:Person)-[:HAS_SECTION]->(:OtherSection)
(:Person)-[:ACHIEVED]->(:Achievement)
(:Person)-[:HAS_SKILL]->(:Skill)
(:Person)-[:WORKED_AT]->(:Company)
(:Person)-[:HELD_ROLE]->(:Role)
(:Role)-[:USED_SKILL]->(:Skill)
(:Project)-[:USED_SKILL]->(:Skill)
"""

In [94]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

cypher_prompt = ChatPromptTemplate.from_template("""
You are a Neo4j Cypher expert.

Generate a READ-ONLY Cypher query
to answer the user's question.

Graph Schema:
{schema}

Person ID:
{person_id}

Question:
{question}

Rules:
- Use only labels and relationships in the schema.
- Always filter by person_id through the Person node.
- Use $person_id as a query parameter.
- Do not use CREATE, MERGE, DELETE, SET, or DROP.
- Return relevant properties and relationships.
- Return only the Cypher query.

Cypher:
""")

cypher_chain = cypher_prompt | llm | StrOutputParser()

In [103]:
cypher = cypher_chain.invoke({
    "schema": schema,
    "person_id": "person_234",
    "question": "what  skills this person has?"
})

In [101]:
cypher = cypher_chain.invoke({
    "schema": schema,
    "person_id": "person_123",
    "question": "What skills this person used in retail forecasting project?"
})

print("Generated Cypher:")
print(cypher)

# Remove markdown formatting
cypher = cypher.replace("```cypher", "").replace("```", "").strip()

print("Clean Cypher:")
print(cypher)

results = graph.query(
    cypher,
    params={"person_id": "person_123"}
)

print(results)

Generated Cypher:
```cypher
MATCH (p:Person {id: $person_id})-[:WORKED_ON]->(proj:Project {name: "retail forecasting"})
MATCH (proj)-[:USED_SKILL]->(skill:Skill)
RETURN DISTINCT skill.name AS skill
```
Clean Cypher:
MATCH (p:Person {id: $person_id})-[:WORKED_ON]->(proj:Project {name: "retail forecasting"})
MATCH (proj)-[:USED_SKILL]->(skill:Skill)
RETURN DISTINCT skill.name AS skill
[]
